# 06: Titles with local pretrained LLMs

**Why:** mT5's Spanish titles were not usable on the owner's episode (05). The owner approved comparing local pretrained instruction LLMs as a replacement. Plan, candidates and selection rule are in the README ("plan for 06").

**Setup:**
- llama.cpp (`llama-cpp-python`, CPU build), GGUF Q4_K_M
- one call per chapter; the same prompt for every model (`lib/llm_titles.py`)
- greedy decoding, one sampled retry, then fallback to mT5

**Where things are evaluated** (so model selection never looks at test):
1. **Validation sample**, for the gates: 150 `val` (en) + all `es_val` (es) pairs, with ground-truth boundaries. Reported: B1/B4/CIDEr, language accuracy, fallback rate, repetition, length, CPU speed.
2. **The owner's episode:** titles from every method for the 6 BiLSTM chapters, written to a **blind sheet** (shuffled, unlabeled). The key is in a separate file.
3. **Test:** only the winner, next to mT5, on the same 800/620-pair sample as 05. Run once, after the owner picks, in the last section.

In [1]:
import json, random, time, subprocess, os
from pathlib import Path
import pandas as pd, psutil
from dotenv import load_dotenv
load_dotenv('../.env')
from lib import data as d, vidchapters as vc, titles as T, llm_titles as L
from lib.units import Atom

ROOT = d.data_root(); RUN = Path('runs/2026-09-28-titles-llm'); RUN.mkdir(parents=True, exist_ok=True)
LLM_DIR = ROOT / 'llm'
MODELS = {
    'qwen3.5-2b': 'Qwen3.5-2B-Q4_K_M.gguf',
    'qwen3.5-4b': 'Qwen3.5-4B-Q4_K_M.gguf',
    'gemma4-e2b': 'gemma-4-E2B-it-Q4_K_M.gguf',
    'gemma4-e4b': 'gemma-4-E4B-it-Q4_K_M.gguf',
    'salamandra-7b': 'salamandra-7b-instruct-2606.Q4_K_M.gguf',
    'qwen3.5-9b': 'Qwen3.5-9B-Q4_K_M.gguf',
}
THREADS = 6            # physical cores; production-realistic, CPU only
MAX_WORDS = 900        # chapter text budget per call (~1.2-1.5k tokens)
lid = d.LanguageId(ROOT.parent / 'models' / 'lid.176.ftz')
MT5 = ROOT / 'checkpoints' / 'titles' / 'best'
def wait_for(path, poll=60):
    # the GGUFs download in MODELS order; hf_hub_download only renames into place when complete
    while not path.exists(): time.sleep(poll)
    return path

def full_pairs(split):
    # like T.make_pairs, but without the 2000-char cap: the LLMs get the whole chapter (sample_text caps it)
    out = []
    for v in vc.load_jsonl(ROOT / f'interim/{split}.jsonl'):
        for i, raw in enumerate(v.chapter_titles):
            title, text = T.clean_title(raw), T.chapter_text(v, i, max_chars=10**7)
            if title and len(text) >= 40:
                out.append({'vid': v.vid, 'lang': v.lang, 'i': i, 'text': text, 'title': title})
    return out
rng = random.Random(2)
val_en = rng.sample(full_pairs('val'), 150)
val_es = full_pairs('es_val')
SAMPLE = {'val_en': val_en, 'val_es': val_es}
print({k: len(v) for k, v in SAMPLE.items()}, 'median words/chapter:',
      {k: int(pd.Series([len(p['text'].split()) for p in v]).median()) for k, v in SAMPLE.items()})

/home/pablo/projects/new_youtube_suite/backend/.venv/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


{'val_en': 150, 'val_es': 117} median words/chapter: {'val_en': 342, 'val_es': 178}


## 1. References on the same sample: mT5 (as shipped in 05) and `lead-8`

In [2]:
import torch
tok, mt5 = T.load_model('cuda' if torch.cuda.is_available() else 'cpu', MT5)
hyps = {}
for name, ps in SAMPLE.items():
    hyps[('mt5-small', name)] = T.generate(tok, mt5, [p['text'][:2000] for p in ps], mt5.device.type)
    hyps[('lead-8', name)] = [' '.join(p['text'].split()[:8]) for p in ps]
del mt5; torch.cuda.empty_cache()

Loading weights:   0%|          | 0/190 [00:00<?, ?it/s]

Loading weights:  26%|██▋       | 50/190 [00:00<00:00, 390.87it/s]

Loading weights:  47%|████▋     | 90/190 [00:00<00:00, 288.04it/s]

Loading weights:  63%|██████▎   | 120/190 [00:00<00:00, 262.51it/s]

Loading weights:  77%|███████▋  | 147/190 [00:00<00:00, 236.94it/s]

Loading weights:  91%|█████████ | 172/190 [00:00<00:00, 216.81it/s]

Loading weights: 100%|██████████| 190/190 [00:00<00:00, 222.35it/s]


[transformers] The tied weights mapping and config for this model specifies to tie shared.weight to lm_head.weight, but both are present in the checkpoints with different values, so we will NOT tie them. You should update the config with `tie_word_embeddings=False` to silence this warning.


## 2. Every LLM on the validation sample (CPU only)

Results are cached per model in `RUN/gen_<model>.jsonl`, so a crash or a restart resumes where it stopped.

In [3]:
proc = psutil.Process()
speed = {}
for key, fname in MODELS.items():
    cache = RUN / f'gen_{key}.jsonl'
    done = [json.loads(l) for l in cache.open()] if cache.exists() else []
    todo = [(n, j) for n, ps in SAMPLE.items() for j in range(len(ps))][len(done):]
    if todo:
        wait_for(LLM_DIR / fname); t0 = time.time(); m = L.LlamaTitler(LLM_DIR / fname, n_threads=THREADS, max_words=MAX_WORDS); load_s = time.time() - t0
        with cache.open('a') as f:
            for n, j in todo:
                p = SAMPLE[n][j]; r = m.title(p['text'], p['lang'], lid=lid)
                row = dict(set=n, j=j, title=r.title, raw=r.raw, seconds=r.seconds, prompt_tokens=r.prompt_tokens,
                           attempts=r.attempts, rss_gb=proc.memory_info().rss / 1e9, load_s=load_s)
                f.write(json.dumps(row, ensure_ascii=False) + '\n'); f.flush(); done.append(row)
        del m
    df = pd.DataFrame(done)
    speed[key] = dict(s_per_chapter=df.seconds.median(), prompt_tokens=df.prompt_tokens.median(),
                      rss_gb=df.rss_gb.max(), load_s=df.load_s.iloc[0])
    for n in SAMPLE:
        sub = df[df.set == n].sort_values('j')
        fb = [t if t else h for t, h in zip(sub.title, hyps[('mt5-small', n)])]   # fallback to mT5, as in production
        hyps[(key, n)] = fb
        speed[key][f'fallback_{n}'] = float(sub.title.isna().mean())
    print(key, {k: round(v, 2) for k, v in speed[key].items()}, flush=True)

qwen3.5-2b {'s_per_chapter': np.float64(6.34), 'prompt_tokens': np.float64(405.0), 'rss_gb': np.float64(3.96), 'load_s': np.float64(1.3), 'fallback_val_en': 0.0, 'fallback_val_es': 0.0}


qwen3.5-4b {'s_per_chapter': np.float64(16.02), 'prompt_tokens': np.float64(405.0), 'rss_gb': np.float64(6.43), 'load_s': np.float64(35.61), 'fallback_val_en': 0.0, 'fallback_val_es': 0.0}


gemma4-e2b {'s_per_chapter': np.float64(8.26), 'prompt_tokens': np.float64(398.0), 'rss_gb': np.float64(5.39), 'load_s': np.float64(5.75), 'fallback_val_en': 0.0, 'fallback_val_es': 0.0}


gemma4-e4b {'s_per_chapter': np.float64(15.47), 'prompt_tokens': np.float64(398.0), 'rss_gb': np.float64(8.57), 'load_s': np.float64(28.3), 'fallback_val_en': 0.0, 'fallback_val_es': 0.0}


salamandra-7b {'s_per_chapter': np.float64(20.48), 'prompt_tokens': np.float64(412.0), 'rss_gb': np.float64(10.13), 'load_s': np.float64(40.74), 'fallback_val_en': 0.0, 'fallback_val_es': 0.0}


qwen3.5-9b {'s_per_chapter': np.float64(26.24), 'prompt_tokens': np.float64(405.0), 'rss_gb': np.float64(10.37), 'load_s': np.float64(32.27), 'fallback_val_en': 0.0, 'fallback_val_es': 0.0}


## 3. Validation metrics and gates

In [4]:
def raw_repeats(t):
    return L.collapse_repeats(t) != ' '.join(t.split())
rows = []
for (method, n), hs in hyps.items():
    ps = SAMPLE[n]
    lang = ps[0]['lang']
    rows.append(dict(method=method, set=n, **T.score([p['title'] for p in ps], hs),
                     lang_ok=100 * (1 - sum(L.wrong_language(h, lang, lid) for h in hs) / len(hs)),
                     repeats=100 * sum(raw_repeats(h) for h in hs) / len(hs),
                     words=sum(len(h.split()) for h in hs) / len(hs)))
val = pd.DataFrame(rows)
table = val.pivot(index='method', columns='set', values=['CIDEr', 'B1', 'lang_ok', 'repeats', 'words']).round(1)
sp = pd.DataFrame(speed).T
EP_CHAPTERS = 6
sp['episode_min_est'] = (sp.s_per_chapter * EP_CHAPTERS * 1.0 / 60).round(1)
gates = pd.DataFrame({
    'lang_ok_es>=95': table[('lang_ok', 'val_es')] >= 95,
    'CIDEr_en>=0.8*mt5': table[('CIDEr', 'val_en')] >= 0.8 * table.loc['mt5-small', ('CIDEr', 'val_en')],
}).join(sp.episode_min_est.le(15).rename('cpu<=15min'), how='left')
display(table); display(sp.round(2)); display(gates)

CIDEr            B1        lang_ok        repeats         \
set           val_en val_es val_en val_es  val_en val_es  val_en val_es   
method                                                                    
gemma4-e2b      47.7   27.9   13.4    7.7   100.0  100.0     0.0    0.0   
gemma4-e4b      49.3   53.8   13.2   11.8   100.0  100.0     0.0    0.0   
lead-8          31.9   32.8    7.3    7.9   100.0   87.2     7.3   24.8   
mt5-small       65.9   50.0   12.9   12.3    99.3   90.6     6.0    6.8   
qwen3.5-2b      57.8   47.6   16.6   12.0   100.0  100.0     0.0    0.0   
qwen3.5-4b      55.9   44.5   13.8   10.5   100.0  100.0     0.0    0.0   
qwen3.5-9b      46.1   54.5   12.8   11.1   100.0  100.0     0.0    0.0   
salamandra-7b   51.6   47.2   12.4   12.2   100.0  100.0     0.0    0.0   

               words         
set           val_en val_es  
method                       
gemma4-e2b       6.5    8.5  
gemma4-e4b       7.0    7.4  
lead-8           8.0    8.0  
mt5-small        5.6    4.8  
qwen3.5-2b       4.7    4.9  
qwen3.5-4b       7.3    8.5  
qwen3.5-9b       7.1    7.1  
salamandra-7b    3.8    4.9

,s_per_chapter,prompt_tokens,rss_gb,load_s,fallback_val_en,fallback_val_es,episode_min_est
qwen3.5-2b,6.34,405.0,3.96,1.30,0.0,0.0,0.6
qwen3.5-4b,16.02,405.0,6.43,35.61,0.0,0.0,1.6
gemma4-e2b,8.26,398.0,5.39,5.75,0.0,0.0,0.8
gemma4-e4b,15.47,398.0,8.57,28.30,0.0,0.0,1.5
salamandra-7b,20.48,412.0,10.13,40.74,0.0,0.0,2.0
qwen3.5-9b,26.24,405.0,10.37,32.27,0.0,0.0,2.6


,lang_ok_es>=95,CIDEr_en>=0.8*mt5,cpu<=15min
method,,,
gemma4-e2b,True,False,True
gemma4-e4b,True,False,True
lead-8,False,False,NaN
mt5-small,False,True,NaN
qwen3.5-2b,True,True,True
qwen3.5-4b,True,True,True
qwen3.5-9b,True,False,True
salamandra-7b,True,False,True


## 4. The owner's episode: blind sheet

Chapter starts are the BiLSTM's from 05. Every method titles every chapter, on CPU, with the timings measured for real. The sheet shuffles the candidates per chapter and doesn't name them. The key is written to `blind_key.json`, which the owner shouldn't open before judging.

In [5]:
e05 = json.load(open('runs/2026-09-28-eval/metrics.json'))
starts = e05['episode']['bilstm']
UPLOAD_ID = 8
q = f'''select coalesce(json_agg(json_build_object('s', w.start_seconds, 'e', w.end_seconds, 't', w.word)
        order by s.segment_index, w.word_index), '[]') from transcript_words w join transcript_segments s on s.id = w.segment_id
        where s.upload_id = {UPLOAD_ID}'''
out = subprocess.run(['docker', 'exec', 'new_youtube_suite-db-1', 'psql', '-U', 'youtube_suite', '-d', 'youtube_suite', '-At', '-c', q],
                     capture_output=True, text=True, check=True).stdout
words = json.loads(out); duration = max(w['e'] or 0 for w in words)
ends = [*starts[1:], duration]
texts = [' '.join(w['t'].strip() for w in words if a <= w['s'] < b) for a, b in zip(starts, ends)]
ep_lang, conf = lid(' '.join(texts)[:5000]); print('episode language:', ep_lang, round(conf, 2), '| words/chapter:', [len(t.split()) for t in texts])
ep = {}
tok, mt5 = T.load_model('cpu', MT5); ep['mt5-small'] = dict(titles=T.generate(tok, mt5, [t[:2000] for t in texts], 'cpu'), seconds=None); del mt5
for key, fname in MODELS.items():
    cache = RUN / f'ep_{key}.json'   # per-model cache: a crash or a restart resumes where it stopped
    if cache.exists():
        ep[key] = json.loads(cache.read_text()); continue
    m = L.LlamaTitler(LLM_DIR / fname, n_threads=THREADS, max_words=MAX_WORDS)
    t0, prev, titles, raws = time.time(), [], [], []
    for t in texts:
        r = m.title(t, ep_lang, prev, lid=lid); titles.append(r.title); raws.append(r.raw)
        if r.title: prev.append(r.title)
    ep[key] = dict(titles=titles, raw=raws, seconds=time.time() - t0)
    cache.write_text(json.dumps(ep[key], ensure_ascii=False))
    del m
    print(key, f"{ep[key]['seconds'] / 60:.1f} min", titles, flush=True)
fmt = lambda t: f'{int(t // 3600)}:{int(t % 3600 // 60):02d}:{int(t % 60):02d}'
rb = random.Random(7); key_map, lines = {}, ['# Blind review: chapter titles for upload 8', '',
    'For each chapter, rank the candidates or mark the ones you would publish. Candidates are shuffled per chapter.', '']
for c, (s, t) in enumerate(zip(starts, texts)):
    cands = [(meth, v['titles'][c] or '(no title)') for meth, v in ep.items()]
    rb.shuffle(cands); letters = 'ABCDEFGHIJ'
    key_map[c] = {letters[k]: meth for k, (meth, _) in enumerate(cands)}
    lines += [f'## Chapter {c + 1}, {fmt(s)}', '', f'> {t[:400]}…', '']
    lines += [f'- **{letters[k]}.** {title}' for k, (_, title) in enumerate(cands)] + ['']
(RUN / 'blind_sheet.md').write_text('\n'.join(lines))
(RUN / 'blind_key.json').write_text(json.dumps(key_map, indent=1))
print('blind sheet ->', RUN / 'blind_sheet.md')

episode language: es 0.98 | words/chapter: [4465, 879, 472, 653, 1592, 3280]


Loading weights:   0%|          | 0/190 [00:00<?, ?it/s]

Loading weights: 100%|██████████| 190/190 [00:00<00:00, 5642.65it/s]


[transformers] The tied weights mapping and config for this model specifies to tie shared.weight to lm_head.weight, but both are present in the checkpoints with different values, so we will NOT tie them. You should update the config with `tie_word_embeddings=False` to silence this warning.


qwen3.5-2b 1.9 min ['Gafas mal ajustadas y polvos', 'La reforma de la justicia y el debate de los', '¡Fallo total en el partido de ayer!', 'El Madrid se va de la liga', 'Boda en la calle del Capitán de Mesa', 'Duda sobre compra del ático']


...

llama_kv_cache: the V embeddings have different sizes across layers and FA is not enabled - padding V cache to 1024


gemma4-e4b 4.7 min ['Ajuste de gafas y secretos de oposiciones revelados', 'Riqueza, oposiciones y el placer de escribir en papel', 'Madrid, Atleti, Mourinho y el experimento de Florentino', 'Madrid debe irse de liga, liga adulterada', 'Fiebre de bodas, monogamia y consejos matrimoniales', 'Ceuta, fútbol, ático y citas literarias en debate']


salamandra-7b 5.1 min ['Ajuste de gafas y consejos de moda', 'Abogados del Estado y el debate de la riqueza', 'Perales y el grito de María Gray', 'Madrid debería irse de la liga', 'Casarse y divorciarse es un juego peligroso', 'Gobierno y competencias en Ceuta']


qwen3.5-9b 6.9 min ["El ajuste de las gafas y el reto de Men's", 'Debate sobre la justicia y el papel', 'Viaje en carretera y debate sobre el partido', 'Madrid debe irse de la liga adulterada', 'Fiebre de bodas y el regreso a la monogamia', 'Ceuta, el ático y la entrevista a Bellerín']


blind sheet -> runs/2026-09-28-titles-llm/blind_sheet.md


In [6]:
json.dump(dict(models=MODELS, threads=THREADS, max_words=MAX_WORDS, n_val_en=len(val_en), n_val_es=len(val_es),
               prompt_version='v1'), open(RUN / 'config.json', 'w'), indent=1)
json.dump(dict(val=json.loads(val.to_json(orient='records')), speed=speed,
               episode={k: dict(seconds=v['seconds']) for k, v in ep.items()}),
          open(RUN / 'metrics.json', 'w'), indent=1)
print('saved to', RUN)

saved to runs/2026-09-28-titles-llm


## 5. Test: the winner next to mT5 (run once)

**The owner picked `qwen3.5-4b`** (2026-09-28). In the blind review they chose qwen3.5-9b's title for all 6 chapters, but the 9B fails the CIDEr_en gate and the ~5 GB RAM budget. Following the selection rule, the pick is among the models that pass the gates (qwen3.5-2b, qwen3.5-4b).

Sample: 05's draws (`random.Random(1)`, up to 800 pairs from `test`, then from `es_test`), **cut to the first 150 of each**. The full 05 sample would take ~10 h on CPU; the owner chose the reduced one (2026-09-28). A prefix of a random sample is still a random sample, and mT5 is re-scored on the same 150 + 150. mT5 gets its 2000-char input as shipped; the LLM gets the whole chapter (capped by `sample_text`), and falls back to mT5 when it gives no title, as in production. Cached per pair in `RUN/test_<model>.jsonl`, so a restart resumes.

In [ ]:
WINNER = 'qwen3.5-4b'
N_TEST = 150   # per split; see above
rng = random.Random(1); TEST = {}
for s in ('test', 'es_test'):
    vids = list(vc.load_jsonl(ROOT / f'interim/{s}.jsonl'))
    ps = T.make_pairs(vids); ps = rng.sample(ps, min(800, len(ps)))   # same draws as 05
    full = {(v.vid, i): T.chapter_text(v, i, max_chars=10**7) for v in vids for i in range(len(v.chapter_titles))}
    TEST[s] = [dict(p, full=full[(p['vid'], p['i'])]) for p in ps[:N_TEST]]
print({k: len(v) for k, v in TEST.items()})

tok, mt5 = T.load_model('cuda' if torch.cuda.is_available() else 'cpu', MT5)
thyps = {('mt5-small', s): T.generate(tok, mt5, [p['text'] for p in ps], mt5.device.type) for s, ps in TEST.items()}
del mt5; torch.cuda.empty_cache()

cache = RUN / f'test_{WINNER}.jsonl'
done = [json.loads(l) for l in cache.open()] if cache.exists() else []
todo = [(s, j) for s, ps in TEST.items() for j in range(len(ps))][len(done):]
if todo:
    m = L.LlamaTitler(LLM_DIR / MODELS[WINNER], n_threads=THREADS, max_words=MAX_WORDS)
    with cache.open('a') as f:
        for s, j in todo:
            p = TEST[s][j]; r = m.title(p['full'], p['lang'], lid=lid)
            row = dict(set=s, j=j, title=r.title, raw=r.raw, seconds=r.seconds, attempts=r.attempts)
            f.write(json.dumps(row, ensure_ascii=False) + '\n'); f.flush(); done.append(row)
    del m
tdf = pd.DataFrame(done)
for s in TEST:
    sub = tdf[tdf.set == s].sort_values('j')
    thyps[(WINNER, s)] = [t if t else h for t, h in zip(sub.title, thyps[('mt5-small', s)])]

trows = []
for (method, s), hs in thyps.items():
    ps = TEST[s]; lang = ps[0]['lang']
    trows.append(dict(method=method, set=s, n=len(ps), **T.score([p['title'] for p in ps], hs),
                      lang_ok=100 * (1 - sum(L.wrong_language(h, lang, lid) for h in hs) / len(hs)),
                      repeats=100 * sum(raw_repeats(h) for h in hs) / len(hs),
                      words=sum(len(h.split()) for h in hs) / len(hs)))
test = pd.DataFrame(trows)
test_table = test.pivot(index='method', columns='set', values=['CIDEr', 'B1', 'B4', 'lang_ok', 'repeats', 'words']).round(1)
display(test_table)
print('fallback to mT5:', {s: float(tdf[tdf.set == s].title.isna().mean()) for s in TEST},
      '| s/chapter median:', round(tdf.seconds.median(), 1))
json.dump(dict(winner=WINNER, sample=f'05 draws (Random(1), <=800 per split), first {N_TEST} of each', test=json.loads(test.to_json(orient='records')),
               fallback={s: float(tdf[tdf.set == s].title.isna().mean()) for s in TEST}, s_per_chapter=float(tdf.seconds.median())),
          open(RUN / 'test_metrics.json', 'w'), indent=1)
print('saved to', RUN / 'test_metrics.json')